# 04 — Content-Based Recommendation

**Goal:** represent movie metadata as TF-IDF vectors and retrieve similar movies using cosine distance. Sparse nearest-neighbour search avoids building a huge dense all-pairs matrix.

## 1. Load processed movie data and imports

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# Keep paths consistent whether the notebook is launched from the project root or 02_Notebooks.
CWD = Path.cwd()
PROJECT_ROOT = CWD.parent if CWD.name == "02_Notebooks" else CWD
DATA_RAW = PROJECT_ROOT / "01_Data" / "Raw"
DATA_PROCESSED = PROJECT_ROOT / "01_Data" / "Processed"
MODELS = PROJECT_ROOT / "03_Models"
VISUALS = PROJECT_ROOT / "05_Visualizations"
for folder in [DATA_RAW, DATA_PROCESSED, MODELS, VISUALS]:
    folder.mkdir(parents=True, exist_ok=True)
print("Project root:", PROJECT_ROOT)

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
import joblib
movies = pd.read_pickle(DATA_PROCESSED / "movies_processed.pkl").reset_index(drop=True)
print("Movie catalogue:", movies.shape)

Project root: D:\DataScience\notebooks\CineMatch_Ready_Notebooks
Movie catalogue: (9742, 12)


## 2. Build TF-IDF feature vectors

TF-IDF gives higher weight to terms that are useful for distinguishing documents and lower weight to terms that appear everywhere. Here, each movie's combined title, genre, and tag text is treated as a document.

In [2]:
tfidf = TfidfVectorizer(
    stop_words="english", ngram_range=(1,2),
    min_df=1, max_features=50000, sublinear_tf=True
)
content_matrix = tfidf.fit_transform(movies["content_text"].fillna(""))
print("TF-IDF matrix:", content_matrix.shape)
print("Non-zero feature values:", content_matrix.nnz)

TF-IDF matrix: (9742, 35275)
Non-zero feature values: 110115


## 3. Fit a cosine nearest-neighbour model

Cosine similarity compares the direction of two vectors. Scikit-learn's cosine distance is `1 - cosine similarity`; we request nearest neighbours without materialising a dense similarity matrix.

In [3]:
content_nn = NearestNeighbors(metric="cosine", algorithm="brute", n_jobs=-1)
content_nn.fit(content_matrix)
print("Content-based nearest-neighbour model fitted.")

Content-based nearest-neighbour model fitted.


## 4. Create robust movie search and recommendation functions

The search function supports partial titles. If multiple movies share a similar title, it returns candidates for the user to distinguish.

In [4]:
def search_movies(query, limit=10):
    query = str(query).strip().lower()
    if not query:
        return movies.iloc[0:0][["movieId","title","genres","average_rating","rating_count"]]
    mask = movies["title"].str.lower().str.contains(query, regex=False, na=False)
    return movies.loc[mask, ["movieId","title","genres","average_rating","rating_count"]].head(limit)

def recommend_similar_movies(movie_title, top_n=10):
    matches = search_movies(movie_title, limit=20)
    if matches.empty:
        return pd.DataFrame(columns=["movieId","title","genres","similarity"])
    if len(matches) > 1:
        print("Multiple matches found. Using the first match; inspect search_movies() to select another.")
        display(matches)
    movie_id = int(matches.iloc[0]["movieId"])
    row_index = int(movies.index[movies["movieId"] == movie_id][0])
    distances, indices = content_nn.kneighbors(content_matrix[row_index], n_neighbors=min(top_n+1, len(movies)))
    results = []
    for distance, idx in zip(distances[0], indices[0]):
        candidate = movies.iloc[int(idx)]
        if int(candidate["movieId"]) == movie_id:
            continue
        results.append({
            "movieId": int(candidate["movieId"]), "title": candidate["title"],
            "genres": candidate["genres"], "similarity": float(1-distance),
            "average_rating": float(candidate["average_rating"]),
            "rating_count": int(candidate["rating_count"])
        })
        if len(results) >= top_n:
            break
    return pd.DataFrame(results)

## 5. Test recommendations

`Toy Story (1995)` is used as an example query. Actual results are generated from the downloaded MovieLens data.

In [5]:
display(search_movies("Toy Story"))
toy_story_recs = recommend_similar_movies("Toy Story (1995)", top_n=10)
display(toy_story_recs)

,movieId,title,genres,average_rating,rating_count
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,3.920930,215
2355,3114,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,3.860825,97
7355,78499,Toy Story 3 (2010),Adventure|Animation|Children|Comedy|Fantasy|IMAX,4.109091,55


,movieId,title,genres,similarity,average_rating,rating_count
0,78499,Toy Story 3 (2010),Adventure|Animation|Children|Comedy|Fantasy|IMAX,0.424833,4.109091,55
1,3114,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,0.392092,3.860825,97
2,2355,"Bug's Life, A (1998)",Adventure|Animation|Children|Comedy,0.283921,3.516304,92
3,13,Balto (1995),Adventure|Animation|Children,0.261937,3.125000,8
4,45074,"Wild, The (2006)",Adventure|Animation|Children|Comedy|Fantasy,0.231144,2.500000,1
5,4886,"Monsters, Inc. (2001)",Adventure|Animation|Children|Comedy|Fantasy,0.224983,3.871212,132
6,53121,Shrek the Third (2007),Adventure|Animation|Children|Comedy|Fantasy,0.221189,3.023810,21
7,103755,Turbo (2013),Adventure|Animation|Children|Comedy|Fantasy,0.220016,2.500000,1
8,2294,Antz (1998),Adventure|Animation|Children|Comedy|Fantasy,0.216590,3.244444,45
9,166461,Moana (2016),Adventure|Animation|Children|Comedy|Fantasy,0.214876,3.450000,10


## 6. Test unknown and ambiguous titles

In [6]:
print("Unknown title test:")
display(recommend_similar_movies("This Movie Does Not Exist"))
print("Ambiguous/partial title test:")
display(search_movies("Batman", limit=10))

Unknown title test:


,movieId,title,genres,similarity


Ambiguous/partial title test:


,movieId,title,genres,average_rating,rating_count
126,153,Batman Forever (1995),Action|Adventure|Comedy|Crime,2.916058,137
509,592,Batman (1989),Action|Crime|Thriller,3.428571,189
1060,1377,Batman Returns (1992),Action|Crime,3.025000,60
1174,1562,Batman & Robin (1997),Action|Adventure|Fantasy|Thriller,2.214286,42
2418,3213,Batman: Mask of the Phantasm (1993),Animation|Children,4.115385,13
5463,26152,Batman (1966),Action|Adventure|Comedy,3.000000,5
5620,27155,"Batman/Superman Movie, The (1998)",Action|Adventure|Animation|Children|Fantasy|Sc...,2.000000,1
5631,27311,Batman Beyond: Return of the Joker (2000),Action|Animation|Crime|Sci-Fi|Thriller,3.500000,3
5917,33794,Batman Begins (2005),Action|Crime|IMAX,3.862069,116
6815,60979,Batman: Gotham Knight (2008),Action|Animation|Crime,3.500000,3


## 7. Save content model artifacts

The fitted vectorizer, sparse feature matrix, movie row order, and nearest-neighbour model must stay aligned. Save them together to preserve that alignment.

In [7]:
content_artifacts = {
    "movies": movies,
    "tfidf_vectorizer": tfidf,
    "content_matrix": content_matrix,
    "content_nn": content_nn
}
joblib.dump(content_artifacts, MODELS / "content_artifacts.joblib", compress=3)
print("Saved:", MODELS / "content_artifacts.joblib")

Saved: D:\DataScience\notebooks\CineMatch_Ready_Notebooks\03_Models\content_artifacts.joblib


## Key concept recap

- **TF-IDF:** turns text into weighted numerical features.
- **Cosine similarity:** measures how similar two feature vectors point.
- **Content-based filtering:** recommends items with metadata similar to an item the user likes.

Next: **05_Collaborative_Filtering.ipynb**.